In [ ]:
import os
from google.colab import drive

# 1. Drive connect
drive.mount('/content/drive')

# 2. Extract the data directly to Colab's fast storage
print("📦 Extracting from Drive to Colab (Isme 2-3 minute lagenge)...")

# Path correction
!unzip -q "/content/drive/MyDrive/SafeStreet_Data.zip" -d "/content/SafeStreet_Dataset"

print("✅ Extraction Complete! Folder structure:")
!ls -l /content/SafeStreet_Dataset

In [ ]:
import os, glob, cv2
import numpy as np
import matplotlib.pyplot as plt

TRAIN_DIR = '/content/SafeStreet_Dataset/SafeStreet_Data/Train'
NUM_FRAMES = 16
IMAGE_SIZE = 224


# 1. Pick up all images (.png and .jpg) from Fighting Folder and sort it
images_path = os.path.join(TRAIN_DIR, 'Fighting', '*.*')
all_frames = sorted([f for f in glob.glob(images_path) if f.endswith(('.png', '.jpg'))])

if len(all_frames) >= NUM_FRAMES:
    print(f"📸 Total frames found in Fighting class: {len(all_frames)}")

    # 2.Take 16 consecutie imaes
    sequence_paths = all_frames[50:50+NUM_FRAMES] #Taking 16 frames from the middle for better action

    print(f"🎬 Creating sequence from: {os.path.basename(sequence_paths[0])}")

    frames = []
    for img_path in sequence_paths:
        frame = cv2.imread(img_path)
        frame = cv2.resize(frame, (IMAGE_SIZE, IMAGE_SIZE))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    # 3. List of images to numpy array (Tensor) 
    frames_array = np.stack(frames)

    print(f"✅ Success! Sequence Shape: {frames_array.shape}")

    # 4. Display the sequence
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for i, ax in enumerate(axes):
        f_idx = i * 4 # Showing frame 0, 4, 8, 12 from our 16-frame chunk
        ax.imshow(frames_array[f_idx])
        ax.set_title(f"Sequence Frame {f_idx}")
        ax.axis('off')
    plt.suptitle("Real Anomaly (Fighting) Frame Sequence", fontsize=14, fontweight='bold')
    plt.show()

else:
    print("❌ Not enough frames found. Please check the dataset path.")

In [ ]:
# ================================================================
# CELL 3 — BALANCED DATA PREPROCESSING & DATALOADER
# ================================================================
import os, glob, cv2, random
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

TRAIN_DIR = '/content/SafeStreet_Dataset/SafeStreet_Data/Train'
TEST_DIR = '/content/SafeStreet_Dataset/SafeStreet_Data/Test'
NUM_FRAMES = 16
IMAGE_SIZE = 224
BATCH_SIZE = 8

class SafeStreetDataset(Dataset):
    # We added max_normal_clips=2500 for balancing
    def __init__(self, root_dir, num_frames=16, transform=None, max_normal_clips=2500):
        self.root_dir = root_dir
        self.num_frames = num_frames
        self.transform = transform
        self.clips = []
        self.labels = []

        self.classes = {'Fighting': 1, 'Assault': 1, 'NormalVideos': 0}
        print(f"🔍 Scanning directory: {root_dir}")

        for cls_name, label in self.classes.items():
            cls_path = os.path.join(root_dir, cls_name)
            if not os.path.exists(cls_path):
                continue

            all_imgs = sorted(glob.glob(os.path.join(cls_path, '*.*')))

            video_groups = {}
            for img in all_imgs:
                base_name = os.path.basename(img).rsplit('_', 1)[0]
                if base_name not in video_groups:
                    video_groups[base_name] = []
                video_groups[base_name].append(img)

            temp_clips = []
            for vid_name, frames in video_groups.items():
                frames = sorted(frames)
                for i in range(0, len(frames) - num_frames + 1, num_frames):
                    temp_clips.append(frames[i : i + num_frames])

            # ⚖️ THE BALANCING MAGIC ⚖️
            if cls_name == 'NormalVideos' and len(temp_clips) > max_normal_clips:
                print(f"  ⚠️ Balancing {cls_name}: Reducing from {len(temp_clips)} down to {max_normal_clips} clips!")
                random.seed(42)
                temp_clips = random.sample(temp_clips, max_normal_clips)

            for clip in temp_clips:
                self.clips.append(clip)
                self.labels.append(label)

            print(f"  📂 {cls_name}: Final Clips Ready -> {len(temp_clips)}")

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_paths = self.clips[idx]
        label = self.labels[idx]
        frames = []
        for p in clip_paths:
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            if self.transform:
                img = self.transform(img)
            frames.append(img)
        return torch.stack(frames), torch.tensor(label, dtype=torch.float32)

video_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\n⚙️ Building Train Dataset...")
train_dataset = SafeStreetDataset(TRAIN_DIR, num_frames=NUM_FRAMES, transform=video_transforms, max_normal_clips=2500)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print("\n⚙️ Building Test Dataset...") # Test data mein max_normal_clips kam rakhenge
test_dataset = SafeStreetDataset(TEST_DIR, num_frames=NUM_FRAMES, transform=video_transforms, max_normal_clips=300)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\n🚀 Balanced DataLoader Ready!")
print(f"Total Training Clips: {len(train_dataset)} (Fast & Efficient!)")

In [ ]:
# ================================================================
# CELL 4 — LOADING THE AI BRAIN (VideoMAE) & OPTIMIZER
# ================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import VideoMAEForVideoClassification

# ── 1. Device Setup (Checking for GPU) ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Hardware Engine: {device.type.upper()}")
if device.type == 'cuda':
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")

# ── 2. Load Pre-trained VideoMAE Model ──
print("\n🧠 Downloading & Loading VideoMAE Base Model from HuggingFace...")

# We are using 'videomae-base' so that we can use jetson nano (phase-3) work
model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base",
    num_labels=2, # Humari 2 classes hain: Normal (0) aur Anomaly (1)
    ignore_mismatched_sizes=True # Ignore purana 400-class wala head
)

# Send model to GPU
model = model.to(device)
print("✅ Model successfully loaded and moved to GPU!")

# ── 3. Define Optimizer & Loss Function ──
# AdamW Transformer models 
learning_rate = 5e-5
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

# CrossEntropyLoss classification 
criterion = nn.CrossEntropyLoss()

# ── 4. Parameter Count Check ──
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📊 Total Trainable Parameters: {total_params:,}")

print("\n🚀 Ready for the Final Step: The Training Loop!")

In [ ]:
# ================================================================
# CELL 5 — THE TRAINING ENGINE
# ================================================================
import time

EPOCHS = 3
print_every = 50 # Printing after 50 batchs

print(f"🚀 Starting VideoMAE Training for {EPOCHS} Epochs...\n")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train() # Put model for training
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0

    print(f"{"="*40}")
    print(f"🔄 EPOCH {epoch+1}/{EPOCHS} STARTED")
    print(f"{"="*40}")

    for batch_idx, (videos, labels) in enumerate(train_loader):
        # 1. Send data to GPU
        videos = videos.to(device)
        labels = labels.to(device).long() # Labels should be LongTensor for CrossEntropy

        # 2. Forward Pass: Model predictions 
        outputs = model(videos)
        logits = outputs.logits # Prediction probabilities

        # 3. Loss Calculation
        loss = criterion(logits, labels)

        # 4. Backward Pass & Optimization: Correct the mistake
        optimizer.zero_grad() # Clear Previous Gradient
        loss.backward()       # Calculate the new Gradient
        optimizer.step()      # Update the Weights

        # --- Metrics tracking ---
        running_loss += loss.item() * videos.size(0)
        _, predicted = torch.max(logits, 1) # Class with highest probability
        correct_preds += (predicted == labels).sum().item()
        total_samples += labels.size(0)

        # After each 50 batches live updatign
        if (batch_idx + 1) % print_every == 0:
            current_loss = running_loss / total_samples
            current_acc = (correct_preds / total_samples) * 100
            print(f"  ⏳ Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {current_loss:.4f} | Acc: {current_acc:.2f}%")

    # --- End of Epoch Summary ---
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = (correct_preds / len(train_dataset)) * 100

    print(f"\n✅ EPOCH {epoch+1} FINISHED!")
    print(f"📈 Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_acc:.2f}%\n")

total_time = (time.time() - start_time) / 60
print(f"🎉 TRAINING COMPLETE in {total_time:.2f} minutes!")

# ── Save the Model (Bohot Zaroori!) ──
torch.save(model.state_dict(), '/content/SafeStreet_VideoMAE.pth')
print("💾 Model saved as 'SafeStreet_VideoMAE.pth'")

In [ ]:
# ================================================================
# CELL 6 — EVALUATION ON UNSEEN TEST DATA
# ================================================================
import torch
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

print("🕵️‍♀️ Starting Evaluation on Unseen Test Data...")
print("Please wait, analyzing test videos...\n")

# 1. Put the model for Evaluation
model.eval()

correct = 0
total = 0
all_preds = []
all_labels = []

# 2. No_grad() saves memory and makes testing 2x faster
with torch.no_grad():
    for batch_idx, (videos, labels) in enumerate(test_loader):
        videos = videos.to(device)
        labels = labels.to(device).long()

        # Model predictions
        outputs = model(videos)
        logits = outputs.logits
        _, predicted = torch.max(logits, 1)

        # Track accuracy
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Store for detailed report
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 3. Final Overall Accuracy
test_acc = (correct / total) * 100
print(f"{"="*40}")
print(f"🎯 FINAL TEST ACCURACY: {test_acc:.2f}%")
print(f"{"="*40}\n")

# 4. Detailed Classification Report 
target_names = ['Normal (0)', 'Anomaly/Crime (1)']
print("📊 Detailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=target_names))

# 5. Visualizing the Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted by SafeStreet AI', fontsize=12)
plt.ylabel('Actual Label', fontsize=12)
plt.title('Confusion Matrix: Model Performance', fontsize=14, fontweight='bold')
plt.show()